In [1]:
# Importing Libraries

import os
import time
import pandas as pd

from dotenv import load_dotenv


# LangChain Imports

from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.agents import Tool, AgentType,initialize_agent
from langchain.tools import StructuredTool
from langchain.callbacks import get_openai_callback
from pydantic import BaseModel



# Memory Imports

from langchain.memory import (
ConversationBufferMemory,
ConversationBufferWindowMemory,
ConversationSummaryMemory,
ConversationTokenBufferMemory,
CombinedMemory,
VectorStoreRetrieverMemory
)

# Vector Store Import

from langchain.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings

# Tavily Search Import

from langchain_community.tools.tavily_search import TavilySearchResults

In [ ]:
# Load Environment Variables

load_dotenv(".env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")


# LLM

llm = ChatOpenAI(model="gpt-3.5-turbo",temperature=0)

### Tools

In [3]:
# TOOL 1 : QA Tool

qa_prompt = PromptTemplate.from_template(
"Answer clearly: {question}"
)

qa_chain = LLMChain(
llm=llm,
prompt=qa_prompt
)

qa_tool = Tool(
name="Simple_QA",
func=qa_chain.run,
description="Answer factual questions clearly."
)

# TOOL 2 : Tavily Search

search_tool = Tool(
name="Web_Search",
func=TavilySearchResults(max_results=3).run,
description="Search current information from the internet."
)


# TOOL 3 : Structured Tool

class TitleInput(BaseModel):
    topic: str
    tone: str = "scholarly"

def title_tool_fn(
    topic: str,
    tone: str = "scholarly"
    ):
    return f"{tone.title()} Title: {topic}"

title_tool = StructuredTool.from_function(
name="TitleMaker",
func=title_tool_fn,
description="Generate scholarly title.",
args_schema=TitleInput
)

/var/folders/07/ykgp85052b11h5kz22ghn8l40000gn/T/ipykernel_79765/229873353.py:7: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  qa_chain = LLMChain(
/var/folders/07/ykgp85052b11h5kz22ghn8l40000gn/T/ipykernel_79765/229873353.py:22: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-tavily package and should be used instead. To use it run `pip install -U :class:`~langchain-tavily` and import as `from :class:`~langchain_tavily import TavilySearch``.
  func=TavilySearchResults(max_results=3).run,


In [4]:
# Memory Factory Function

def get_memory(memory_name):

    if memory_name == "ConversationBufferMemory":

        return ConversationBufferMemory(
            memory_key="chat_history",
            return_messages=True
        )

    elif memory_name == "ConversationBufferWindowMemory":

        return ConversationBufferWindowMemory(
            k=3,
            memory_key="chat_history"
        )

    elif memory_name == "ConversationSummaryMemory":

        return ConversationSummaryMemory(
            llm=llm,
            memory_key="chat_history"
        )

    elif memory_name == "ConversationTokenBufferMemory":

        return ConversationTokenBufferMemory(
            llm=llm,
            memory_key="chat_history",
            max_token_limit=500
        )

    elif memory_name == "CombinedMemory":

        memory1 = ConversationBufferMemory(
            memory_key="chat_history"
        )

        memory2 = ConversationSummaryMemory(
            llm=llm,
            memory_key="summary"
        )

        return CombinedMemory(
            memories=[memory1, memory2]
        )

    elif memory_name == "VectorStoreRetrieverMemory":

        embeddings = OpenAIEmbeddings()

        retriever = FAISS.from_texts(
            ["Initial Memory"],
            embeddings
        ).as_retriever()

        return VectorStoreRetrieverMemory(
            retriever=retriever,
            memory_key="chat_history"
        )
        
# Agent Definitions


agent_configs = {
    "ZERO_SHOT_REACT_DESCRIPTION":
        AgentType.ZERO_SHOT_REACT_DESCRIPTION,

    "CONVERSATIONAL_REACT_DESCRIPTION":
        AgentType.CONVERSATIONAL_REACT_DESCRIPTION,

    "CHAT_CONVERSATIONAL_REACT_DESCRIPTION":
        AgentType.CHAT_CONVERSATIONAL_REACT_DESCRIPTION,

    "STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION":
        AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,

    "OPENAI_FUNCTIONS":
        AgentType.OPENAI_FUNCTIONS,

    "OPENAI_MULTI_FUNCTIONS":
        AgentType.OPENAI_MULTI_FUNCTIONS
}


# Memory Types

memory_types = [
    "ConversationBufferMemory",
    "ConversationBufferWindowMemory",
    "ConversationSummaryMemory",
    "ConversationTokenBufferMemory",
    "CombinedMemory",
    "VectorStoreRetrieverMemory"
]

In [ ]:
# Results Storage

results = []

# Run Matrix

for agent_name, agent_type in agent_configs.items():


    for memory_name in memory_types:
    
        print(
            f"\nTesting -> {agent_name}"
            f" + {memory_name}"
        )

        try:
    
            memory = get_memory(memory_name)
    
            # ----------------------------------
            # Agent-Specific Tools
            # ----------------------------------
    
            if (
                agent_name
                ==
                "STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION"
            ):
    
                tools = [title_tool]
    
                query = (
                    "Create a scholarly title "
                    "for Dark Matter"
                )
    
            else:
    
                tools = [
                    qa_tool,
                    search_tool
                ]
    
                query = (
                    "What is Dark Matter? "
                    "Answer in 2 lines."
                )
    
            # ----------------------------------
            # Agent Creation
            # ----------------------------------
    
            start_time = time.time()
    
            agent = initialize_agent(
                tools=tools,
                llm=llm,
                memory=memory,
                agent=agent_type,
                verbose=False,
                handle_parsing_errors=True
            )
    
            with get_openai_callback() as cb:
    
                response = agent.run(query)
    
                prompt_tokens = cb.prompt_tokens
                completion_tokens = cb.completion_tokens
                total_tokens = cb.total_tokens
    
            execution_time = round(
                time.time() - start_time,
                2
            )
    
            results.append({
    
                "Agent_Type":
                    agent_name,
    
                "Memory_Type":
                    memory_name,
    
                "Compatibility":
                    "Supported",
    
                "Status":
                    "Success",
    
                "Execution_Time_Sec":
                    execution_time,
    
                "Prompt_Tokens":
                    prompt_tokens,
    
                "Completion_Tokens":
                    completion_tokens,
    
                "Total_Tokens":
                    total_tokens,
    
                "Response":
                    response,
    
                "Error":
                    ""
    
            })
    
            print("SUCCESS")

        except Exception as e:
    
            results.append({
    
                "Agent_Type":
                    agent_name,
    
                "Memory_Type":
                    memory_name,
    
                "Compatibility":
                    "Unsupported",
    
                "Status":
                    "Failed",
    
                "Execution_Time_Sec":
                    None,
    
                "Prompt_Tokens":
                    None,
    
                "Completion_Tokens":
                    None,
    
                "Total_Tokens":
                    None,
    
                "Response":
                    "",
    
                "Error":
                    str(e)
    
            })
    
            print("FAILED")


In [6]:
# Create DataFrame

df = pd.DataFrame(results)

# Save CSV

output_file = "Agent_Memory_Comparison.csv"

df.to_csv(output_file, index=False)

print("\n================================================")
print("CSV Export Complete")
print("Rows Generated :", len(df))
print("File :", output_file)
print("================================================")

# Optional Preview

print(df.head())


CSV Export Complete
Rows Generated : 36
File : Agent_Memory_Comparison.csv
                    Agent_Type                     Memory_Type Compatibility  \
0  ZERO_SHOT_REACT_DESCRIPTION        ConversationBufferMemory     Supported   
1  ZERO_SHOT_REACT_DESCRIPTION  ConversationBufferWindowMemory     Supported   
2  ZERO_SHOT_REACT_DESCRIPTION       ConversationSummaryMemory     Supported   
3  ZERO_SHOT_REACT_DESCRIPTION   ConversationTokenBufferMemory     Supported   
4  ZERO_SHOT_REACT_DESCRIPTION                  CombinedMemory   Unsupported   

    Status  Execution_Time_Sec  Prompt_Tokens  Completion_Tokens  \
0  Success                7.71         3975.0              171.0   
1  Success                6.86         3981.0              200.0   
2  Success                3.79         2305.0              170.0   
3  Success                7.40         3981.0              200.0   
4   Failed                 NaN            NaN                NaN   

   Total_Tokens                   

In [7]:
print(df.shape)

(36, 10)


In [8]:
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
print("\nColumn Names:")
print(df.columns.tolist())

Rows: 36
Columns: 10

Column Names:
['Agent_Type', 'Memory_Type', 'Compatibility', 'Status', 'Execution_Time_Sec', 'Prompt_Tokens', 'Completion_Tokens', 'Total_Tokens', 'Response', 'Error']


In [9]:
df

,Agent_Type,Memory_Type,Compatibility,Status,Execution_Time_Sec,Prompt_Tokens,Completion_Tokens,Total_Tokens,Response,Error
0,ZERO_SHOT_REACT_DESCRIPTION,ConversationBufferMemory,Supported,Success,7.71,3975.0,171.0,4146.0,Dark matter is a hypothetical form of matter t...,
1,ZERO_SHOT_REACT_DESCRIPTION,ConversationBufferWindowMemory,Supported,Success,6.86,3981.0,200.0,4181.0,Dark matter is a hypothetical form of matter t...,
2,ZERO_SHOT_REACT_DESCRIPTION,ConversationSummaryMemory,Supported,Success,3.79,2305.0,170.0,2475.0,Dark Matter is an invisible and hypothetical f...,
3,ZERO_SHOT_REACT_DESCRIPTION,ConversationTokenBufferMemory,Supported,Success,7.40,3981.0,200.0,4181.0,Dark matter is a hypothetical form of matter t...,
4,ZERO_SHOT_REACT_DESCRIPTION,CombinedMemory,Unsupported,Failed,NaN,NaN,NaN,NaN,,"One input key expected got ['input', 'summary']"
5,ZERO_SHOT_REACT_DESCRIPTION,VectorStoreRetrieverMemory,Supported,Success,3.48,2111.0,129.0,2240.0,Dark Matter is an invisible and hypothetical f...,
6,CONVERSATIONAL_REACT_DESCRIPTION,ConversationBufferMemory,Supported,Success,2.82,900.0,141.0,1041.0,Dark matter is a hypothetical form of matter t...,
7,CONVERSATIONAL_REACT_DESCRIPTION,ConversationBufferWindowMemory,Supported,Success,3.07,896.0,158.0,1054.0,Dark matter is a hypothetical form of matter t...,
8,CONVERSATIONAL_REACT_DESCRIPTION,ConversationSummaryMemory,Supported,Success,5.77,1102.0,208.0,1310.0,Dark matter is a hypothetical form of matter t...,
9,CONVERSATIONAL_REACT_DESCRIPTION,ConversationTokenBufferMemory,Supported,Success,4.27,896.0,158.0,1054.0,Dark Matter is a hypothetical form of matter t...,
